# UR5e VLA bed — probe session on existing checkpoints (Kaggle, free)
Inputs: dataset **vla-bed-v2** and the **train notebook's output**. Accelerator GPU T4 x2, Internet ON. No training: the open-loop magnitude probe (predicted vs label magnitudes), then closed-loop probes on the selected checkpoint — nominal with measured magnitudes, clip-to-limit, gain 0.61, temporal ensemble, lighting, target_relocation, float16-VLM consistency — and paired comparisons on the same 100 seeds. Plan of 4 Sep 2026, Step 3.


In [ ]:
# Session A (plan of 4 Sep 2026): probes on the checkpoints that already exist — no training. Inputs: dataset vla-bed-v2 + the train notebook's output.
RUN = "baseline"
RECIPE = "auto"           # "auto" = the recipe of the single vla-bed-<recipe> dataset attached (the frozen suite = its evaluation split); or name it
EPISODES = 100
MAX_HOURS = 7.0
WORKERS = 2
PROBE_CKPTS = ["007500", "010000"]     # selected checkpoint first (R11), final checkpoint second


In [ ]:
import os, subprocess, sys, time, json, pathlib
REPO = "https://github.com/santapong/RoboLLM.git"; BRANCH = "experiment/ur5e-vla-bed"
ROOT = pathlib.Path("/kaggle/working/RoboLLM")
# Kaggle mounts the uploaded zip under /kaggle/input/<slug>/ with or without the zip's top folder; find the manifest.
RECIPE = globals().get("RECIPE", "auto")
found = {json.load(open(p)).get("recipe"): p.parent for p in pathlib.Path("/kaggle/input").rglob("manifest.json") if (p.parent / "train").is_dir()}
if RECIPE == "auto":   # exactly one bed dataset attached → its recipe
    assert len(found) == 1, "attach exactly one vla-bed-<recipe> dataset or set RECIPE; found: " + str(found)
    RECIPE = next(iter(found))
assert RECIPE in found, f"add the private dataset vla-bed-{RECIPE} to this notebook (Add Input); found: " + str(found)
DATA = found[RECIPE]; print("dataset root", DATA, "recipe", RECIPE)
if not ROOT.exists():
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO, str(ROOT)], check=True)
os.chdir(ROOT)
link = ROOT / "datasets" / "vla-bed" / RECIPE; link.parent.mkdir(parents=True, exist_ok=True)
MANIFEST = f"datasets/vla-bed/{RECIPE}/manifest.json"   # the frozen suite: v3 records the same 100 evaluation seeds/targets as v2
if not link.exists(): link.symlink_to(DATA)          # every default path in the bed now resolves to the uploaded data
MEN = ROOT / "sim" / "vla-bed" / "assets" / "mujoco_menagerie"   # robot models are not vendored (BSD notices in NOTICES.md); pinned sparse clone, as scripts/pi_setup.sh does
if not (MEN / ".git").exists():
    subprocess.run(["git", "clone", "--quiet", "--filter=blob:none", "--no-checkout", "https://github.com/google-deepmind/mujoco_menagerie.git", str(MEN)], check=True)
    subprocess.run(["git", "-C", str(MEN), "sparse-checkout", "set", "universal_robots_ur5e", "robotiq_2f85"], check=True)
subprocess.run(["git", "-C", str(MEN), "checkout", "--quiet", "e4049d0a3bfd58d2a3081614e6777d4007e3f86a"], check=True)
print("menagerie", subprocess.run(["git", "-C", str(MEN), "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "sim/vla-bed/requirements-record.txt", "mujoco==3.10.0", "pyyaml", "av"], check=True)  # the bed's physics is not in LeRobot's extras
subprocess.run("apt-get install -y -qq libosmesa6 > /dev/null 2>&1 || true", shell=True)   # MuJoCo fallback renderer
os.environ["MUJOCO_GL"] = "egl"; os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip())
import torch; print("torch", torch.__version__, "cuda", torch.cuda.is_available(), "bf16 native", torch.cuda.is_bf16_supported())
print("commit", subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip())

In [ ]:
# Which renderer works here? EGL (NVIDIA) first, OSMesa second.
import os, subprocess, sys
def try_gl(backend):
    r = subprocess.run([sys.executable, "-c", "import mujoco,numpy as np; m=mujoco.MjModel.from_xml_string('<mujoco><worldbody><geom size=\"1\"/></worldbody></mujoco>'); d=mujoco.MjData(m); r=mujoco.Renderer(m,64,64); r.update_scene(d); print(r.render().mean())"], env={**os.environ, "MUJOCO_GL": backend}, capture_output=True, text=True)
    return r.returncode == 0, (r.stdout + r.stderr).strip()[-200:]
for b in ("egl", "osmesa"):
    ok, msg = try_gl(b); print(b, "OK" if ok else "FAIL", msg if not ok else "")
    if ok: os.environ["MUJOCO_GL"] = b; break
print("MUJOCO_GL =", os.environ["MUJOCO_GL"])

In [ ]:
# Find the training output zip under /kaggle/input and unpack it OUTSIDE /kaggle/working — the Output tab would otherwise carry every checkpoint twice (8.24 GB on 4 Sep 2026).
import glob, zipfile, pathlib, shutil, os, json
zips = glob.glob(f"/kaggle/input/**/vla-bed-{RUN}-output" + ("" if RECIPE == "v2" else f"-{RECIPE}") + ".zip", recursive=True)
assert zips, "attach the train notebook's output (Add Input → Notebook output) — found: " + str(glob.glob("/kaggle/input/**/*.zip", recursive=True)[:10])
UNPACK = pathlib.Path("/tmp/vla-bed-unpacked"); shutil.rmtree(UNPACK, ignore_errors=True); shutil.rmtree("/kaggle/working/unpacked", ignore_errors=True)
zipfile.ZipFile(zips[0]).extractall(UNPACK)
CKPTS = {pm.parent.name: str(pm) for pm in sorted(UNPACK.glob(f"artifacts/{RUN}/kaggle/*/pretrained_model")) if pm.parent.name.isdigit()}   # skip LeRobot's `last` alias (a copy of the final step)
print("checkpoints:", sorted(CKPTS))
for rr in sorted(UNPACK.glob("run_record.json")) + sorted(UNPACK.glob("results/p5/*/*/*/nominal.json")):
    d = json.load(open(rr))
    if "steps_done" in d: print("train record:", {k: d.get(k) for k in ("status", "steps_done", "steps_per_s", "wall_s", "gpu", "peak_vram_gb")})
    else:
        lab = d["policy"]["label"]; b = d[lab]; print("quick check", lab, {k: b.get(k) for k in ("n", "success_rate", "ci95_wilson_success", "safety", "progress_mean", "episode_len_mean", "faults")})
for rr in sorted(UNPACK.glob("results/p5/*/*/*/nominal.json")):   # bring the quick-check JSON into this session's results too
    dst_json = pathlib.Path("sim/vla-bed/results/p5") / rr.relative_to(UNPACK / "results" / "p5"); dst_json.parent.mkdir(parents=True, exist_ok=True); shutil.copy(rr, dst_json)


In [ ]:
# Sharded evaluator: WORKERS processes per suite (evaluate.py --shard i/n, one GPU each when available), merged by evaluate.py --merge into the file a single run would write. Wall-clock budget guarded per suite.
import subprocess, sys, os, time, socket, pathlib, json
t0 = time.time(); HOST = socket.gethostname()
GPUS = [g for g in subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.splitlines() if g.startswith("GPU")]
def out_path(label, suffix): return pathlib.Path(f"sim/vla-bed/results/p5/{HOST}/{label}/{suffix}.json")
def suffix_of(extra):
    s = extra.get("variation", "nominal")
    if extra.get("blank"): s += "_blank"
    if extra.get("gain", 1.0) != 1.0: s += f"_gain{extra['gain']:g}"
    if extra.get("post") == "clip": s += "_clip"
    if extra.get("post") == "ensemble": s += f"_ens{extra.get('replan', 5)}"
    if extra.get("vlm_dtype"): s += f"_{extra['vlm_dtype']}"
    return s
def argv_of(ck, label, extra, episodes):
    a = [sys.executable, "sim/vla-bed/evaluate.py", "--policy", "smolvla", "--run", RUN, "--checkpoint", ck, "--episodes", str(episodes), "--label", label, "--variation", extra.get("variation", "nominal"), "--manifest", MANIFEST]
    if extra.get("blank"): a.append("--blank-image")
    if extra.get("gain", 1.0) != 1.0: a += ["--gain", str(extra["gain"])]
    if extra.get("post"): a += ["--post", extra["post"]]
    if extra.get("replan"): a += ["--replan-every", str(extra["replan"])]
    if extra.get("vlm_dtype"): a += ["--vlm-dtype", extra["vlm_dtype"]]
    return a
def ev(ck, label, episodes=None, **extra):
    """One suite: WORKERS shards in parallel, merged; returns the summary dict (None if skipped or failed)."""
    episodes = episodes or EPISODES
    if time.time() - t0 > MAX_HOURS * 3600: print("budget reached, skipping", label, extra); return None
    suffix = suffix_of(extra); final = out_path(label, suffix); final.parent.mkdir(parents=True, exist_ok=True)
    procs, parts, ts = [], [], time.time()
    for i in range(WORKERS):
        part = final.with_name(f"{suffix}_shard{i}of{WORKERS}.json"); parts.append(part)
        env = {**os.environ, "MUJOCO_GL": os.environ.get("MUJOCO_GL", "egl")}
        if GPUS: env["CUDA_VISIBLE_DEVICES"] = str(i % len(GPUS))
        procs.append(subprocess.Popen(argv_of(ck, label, extra, episodes) + ["--shard", f"{i}/{WORKERS}", "--out", str(part)], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env))
    logs = [p.communicate()[0] for p in procs]
    bad = [(i, l[-600:]) for i, (p, l) in enumerate(zip(procs, logs)) if p.returncode]
    if bad: print("FAILED", label, extra, bad, flush=True); return None
    r = subprocess.run([sys.executable, "sim/vla-bed/evaluate.py", "--merge", *map(str, parts), "--out", str(final)], capture_output=True, text=True)
    if r.returncode: print("MERGE FAILED", label, extra, r.stdout[-400:], r.stderr[-400:], flush=True); return None
    for part in parts: part.unlink(missing_ok=True)
    d = json.load(open(final)); b = d[label]
    print(label, extra, {k: b.get(k) for k in ("n", "success_rate", "ci95_wilson_success", "safety", "progress_mean", "episode_len_mean", "rejected_fraction_mean", "cmd_xyz_linf_mean", "cmd_xyz_over_cap_fraction_mean")}, f"suite {(time.time()-ts)/60:.1f} min [{(time.time()-t0)/3600:.2f} h]", flush=True)
    return d


In [ ]:
# Where does evaluation time go on this box? oracle = env only (IK + physics + one render per chunk); smolvla adds the policy. 20 + 3 episodes, a few minutes.
import subprocess, sys, time, json
def timed(argv, key):
    ts = time.time(); r = subprocess.run(argv, capture_output=True, text=True)
    if r.returncode: print("profile failed:", r.stdout[-300:], r.stderr[-300:]); return None, None
    return time.time() - ts, json.load(open(argv[argv.index("--out") + 1]))[key]
w, b = timed([sys.executable, "sim/vla-bed/evaluate.py", "--policy", "oracle", "--episodes", "20", "--manifest", MANIFEST, "--out", "/tmp/profile_oracle.json"], "oracle")
if b: print(f"oracle 20 ep: {w:.0f} s wall, {w / b['frames_total']:.3f} s/frame (env only, {b['frames_total']} frames)")
if CKPTS:
    ck = CKPTS[sorted(CKPTS)[-1]]
    w, b = timed([sys.executable, "sim/vla-bed/evaluate.py", "--policy", "smolvla", "--run", RUN, "--checkpoint", ck, "--episodes", "3", "--label", "profile", "--manifest", MANIFEST, "--out", "/tmp/profile_smolvla.json"], "profile")
    if b: print(f"smolvla 3 ep: {w:.0f} s wall incl. model load, {b['frames_total']} frames, {b['latency_s_mean']:.2f} s per policy call, {b['chunks_mean']:.1f} calls per episode → env ≈ {(w - b['latency_s_mean'] * b['chunks_mean'] * 3) / b['frames_total']:.3f} s/frame upper bound")


In [ ]:
# Value order (the wall-clock guard drops the tail). Every closed-loop row carries the commanded magnitudes (schema v2).
import subprocess, sys, json
sel, fin = PROBE_CKPTS[0], PROBE_CKPTS[-1]
for st in PROBE_CKPTS:   # 1. open-loop magnitude probe against the training labels: bias vs spread (minutes on a GPU)
    r = subprocess.run([sys.executable, "sim/vla-bed/gpu/magnitude_probe.py", "--run", RUN, "--checkpoint", CKPTS[st], "--frames", "500", "--label", f"{RUN}/{st}", "--dataset-root", f"datasets/vla-bed/{RECIPE}/train"], capture_output=True, text=True)
    print(f"magnitude {RUN}/{st}:", r.stdout[-1800:], r.stderr[-500:] if r.returncode else "", flush=True)
ev(CKPTS[sel], f"{RUN}/{sel}")                                  # 2. nominal, selected checkpoint: measured cmd magnitudes + rejected fraction
ev(CKPTS[sel], f"{RUN}/{sel}", post="clip")                      # 3. how much of the gap is the cap alone
ev(CKPTS[fin], f"{RUN}/{fin}", post="clip")
ev(CKPTS[sel], f"{RUN}/{sel}", gain=0.61)                        # 4. gain probe on the selected checkpoint (only the final one had it)
ev(CKPTS[sel], f"{RUN}/{sel}", post="ensemble", replan=5)        # 5. linear temporal ensemble, re-plan every 5 (Lazzati et al. 2608.02547)
for var in ("lighting", "target_relocation"): ev(CKPTS[sel], f"{RUN}/{sel}", variation=var)   # 6. the two variations never run
ev(CKPTS[sel], f"{RUN}/{sel}", episodes=20, vlm_dtype="float16")  # 7. train/eval dtype consistency, 20 episodes


In [ ]:
# Paired comparisons on the same seeds (compare.py): each probe against the selected checkpoint's nominal suite.
import subprocess, sys, pathlib, socket
host = socket.gethostname(); base = pathlib.Path(f"sim/vla-bed/results/p5/{host}/{RUN}/{sel}")
nominal = base / "nominal.json"
if nominal.exists():
    for name in ("nominal_clip", "nominal_gain0.61", "nominal_ens5", "lighting", "target_relocation"):
        f = base / f"{name}.json"
        if f.exists():
            r = subprocess.run([sys.executable, "sim/vla-bed/compare.py", str(nominal), str(f), "--out", str(base / f"compare_nominal_vs_{name}.json")], capture_output=True, text=True)
            print(r.stdout.strip()[-600:], r.stderr[-300:] if r.returncode else "")
    other = pathlib.Path(f"sim/vla-bed/results/p5/{host}/{RUN}/{fin}/nominal_clip.json")
    if other.exists() and (base / "nominal_clip.json").exists():
        print(subprocess.run([sys.executable, "sim/vla-bed/compare.py", str(base / "nominal_clip.json"), str(other), "--out", str(base / f"compare_clip_{sel}_vs_{fin}.json")], capture_output=True, text=True).stdout.strip()[-600:])


In [ ]:
import shutil, pathlib, hashlib, socket
host = socket.gethostname(); stage = pathlib.Path("/kaggle/working/pack"); shutil.rmtree(stage, ignore_errors=True)
shutil.copytree(f"sim/vla-bed/results/p5/{host}", stage / "results" / "p5" / host)
out = shutil.make_archive(f"/kaggle/working/vla-bed-{RUN}-probes" + ("" if RECIPE == "v2" else f"-{RECIPE}"), "zip", stage); shutil.rmtree(stage)
print(out, round(pathlib.Path(out).stat().st_size / 1e6, 2), "MB", "sha256", hashlib.sha256(open(out, "rb").read()).hexdigest())
print("workstation: sim/vla-bed/gpu/kaggle_import.sh <zip> <sha256>")
